# Train a From-Scratch EW-Mode RoughNet Model (Tuktoyaktuk)

Genuinely new model, randomly-initialized weights -- not a fine-tune of
`09`'s IW-trained checkpoint. Same architecture, same real-attrs
conditioning, same spatial-block leakage-safe split methodology as
`pcrtc/09`, but conditioned on the newly-extracted EW-mode Sentinel-1
data (`s1_patches_tuk_ew`, 1676/1676 patches matched, HH+HV, 6x6 native
pixels at 40m resolution) instead of IW/PC-RTC.

**Uses the identical spatial-block split as `09`/`10`** (same
`LIDAR_DIR`, same `BLOCK_SIZE_M`/`BUFFER_M`/`SEED`) -- since the
underlying LiDAR patches are exactly the same set, this produces the
exact same train/val assignment, making the eventual IW-vs-EW
comparison genuinely apples-to-apples: same patches, same split, only
the conditioning data source differs.

**Resolution caveat, worth keeping in mind when interpreting results**:
EW's native 6x6 pixels per patch (vs IW's 26x26) means far less real
spatial detail survives the bilinear upsample to 256x256. If this model
underperforms `09`, that could reflect EW's coarser resolution as much
as (or more than) anything about HH/HV backscatter being less
informative than VV/VH -- the two are confounded in this single
comparison and can't be cleanly separated without a matched-resolution
control, which is out of scope given the time available.

## GPU configuration

In [ ]:
# Require CUDA -- this notebook trains from scratch, so needs the GPU for the full run
import os
import sys
import json
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import rasterio

assert torch.cuda.is_available(), 'CUDA is required. Run this notebook on the GPU environment.'
DEVICE = torch.device('cuda')
torch.backends.cudnn.benchmark = True
print('GPU:', torch.cuda.get_device_name(0))

## Paths and training configuration

In [ ]:
# Same LIDAR_DIR and split parameters as 09/10 -- only S1_DIR (EW instead of PC-RTC) and the checkpoint name differ
WORKING_REPO = Path('/cs/student/project_msc/2025/aibh/jiayiche')
TESSA_REPO = Path('/cs/student/project_msc/2025/aibh/jiayiche/tessa_baseline')
REGION = 'tuk'
LIDAR_DIR = WORKING_REPO / 'input_data' / 'lidar_patches_tuk_tessa'
S1_DIR = WORKING_REPO / 'input_data' / 's1_patches_tuk_ew'
CHECKPOINT_DIR = WORKING_REPO / 'checkpoints'
OUTPUT_DIR = WORKING_REPO / 's1_training_outputs'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CONTEXT_K = 3
TARGET_HW = (256, 256)
BATCH_SIZE = 8
EPOCHS = 100
TIMESTEPS = 1000
LEARNING_RATE = 1e-4
VAL_FRACTION = 0.15
SEED = 42
NOISE_SCHEDULE = 'linear'
ATTENTION_VARIANT = 'default'
LIDAR_SURVEY_DATE = __import__('datetime').date(2024, 4, 16)

# Spatial-block split parameters -- identical to 09/10, so train/val assignment matches exactly
BLOCK_SIZE_M = 1024.0
BUFFER_M = 150.0

## Import Tessa's baseline implementation

In [ ]:
sys.path.insert(0, str(TESSA_REPO))
from src.model.unet import ConditionalUNet
from src.diffusion.scheduler import LinearDiffusionScheduler, CosineDiffusionScheduler
from src.diffusion.sampling import p_sample_loop_ddpm, p_sample_loop_ddim, p_sample_loop_plms
from src.utils.recon_metrics import rmse, bias, sigma_error, normal_angle_error, average_jsd_multiscale, log_psd_rmse, zncc

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

## EW-mode dataset adapter

Same real-attrs structure as `09`, but the Sentinel-1 side reads a
2-band (HH, HV) 6x6-pixel EW patch instead of a 2-band (VV, VH)
26x26-pixel IW patch. HH/HV are already linear sigma0 (calibrated during
extraction) -- converted to dB the same way VV/VH always have been, then
repeated to 4 channels per view and bilinearly upsampled to 256x256.

In [ ]:
# Real-attrs builder, identical structure to 09's, using Tuktoyaktuk's own survey date
def build_real_attrs(s1_path, times, context_k):
    attrs_path = s1_path / 'attrs.json'
    attrs_list = json.load(open(attrs_path)) if attrs_path.exists() else []
    vecs = []
    for time_path in times:
        idx = int(time_path.stem[1:])  # 't0' -> 0
        a = attrs_list[idx] if idx < len(attrs_list) else {}
        if a.get('acquisition_date'):
            import datetime as dt
            acq_date = dt.date.fromisoformat(a['acquisition_date'])
            age_days = (acq_date - LIDAR_SURVEY_DATE).days
            age_norm = age_days / 30.0
        else:
            age_norm = 0.0
        orbit_dir = 1.0 if a.get('orbit_direction') == 'ASCENDING' else 0.0
        rel_orbit = (a.get('relative_orbit_number') or 0) / 175.0
        vecs.append([age_norm, orbit_dir, rel_orbit, 0.0, 0.0, 0.0, 0.0, 0.0])
    return torch.tensor(vecs, dtype=torch.float32).flatten()


class LidarEwDataset(Dataset):
    def __init__(self, s1_dir, lidar_dir, patch_ids, context_k=3, target_hw=(256, 256), train=False):
        self.s1_dir = Path(s1_dir)
        self.lidar_dir = Path(lidar_dir)
        self.patch_ids = list(patch_ids)
        self.context_k = context_k
        self.target_hw = target_hw
        self.train = train

    def __len__(self):
        return len(self.patch_ids)

    def __getitem__(self, index):
        patch_id = self.patch_ids[index]
        lidar_path = self.lidar_dir / f'lidar_patch_{patch_id}.tif'
        s1_path = self.s1_dir / f's1_patch_{patch_id}'
        with rasterio.open(lidar_path) as src:
            raw = src.read().astype(np.float32)
        target = raw[0]
        mask = (raw[1] > 0.5) if raw.shape[0] > 1 else np.isfinite(target)
        target = np.nan_to_num(target, nan=0.0, posinf=0.0, neginf=0.0)
        valid_count = max(1, int(mask.sum()))
        patch_mean = float(target[mask].sum() / valid_count)
        target = (target - patch_mean) * mask

        times = sorted(s1_path.glob('t*.tif'))[:self.context_k]
        if len(times) < self.context_k:
            raise RuntimeError(f'{s1_path} has fewer than {self.context_k} Sentinel-1 times')
        views = []
        for time_path in times:
            with rasterio.open(time_path) as src:
                sar = src.read()[:2].astype(np.float32)  # HH, HV, already linear sigma0
            sar = np.nan_to_num(sar, nan=0.0, posinf=0.0, neginf=0.0)
            sar = np.maximum(sar, 1e-12)
            sar = 10.0 * np.log10(sar)
            sar_tensor = torch.from_numpy(sar).unsqueeze(0)
            sar_tensor = F.interpolate(sar_tensor, size=self.target_hw, mode='bilinear', align_corners=False).squeeze(0)
            sar_tensor = sar_tensor.repeat(2, 1, 1)  # [HH,HV,HH,HV], same repeat pattern as VV/VH
            views.append(sar_tensor)
        condition = torch.cat(views, dim=0)
        attrs = build_real_attrs(s1_path, times, self.context_k)
        return {'lidar': torch.from_numpy(target).unsqueeze(0).float(), 'mask': torch.from_numpy(mask), 's1': condition.float(), 'attrs': attrs, 'patch_mean': torch.tensor(patch_mean), 'patch_id': patch_id}

## Spatial-block split -- identical to `09`/`10`

Same `LIDAR_DIR`, same `SEED`/`BLOCK_SIZE_M`/`BUFFER_M` -- produces the
exact same train/val patch assignment as `09`, so the eventual
comparison is on identical validation patches.

In [ ]:
# Spatial-block split, identical logic and parameters to 09/10 -- same LIDAR_DIR means the same patches get the same assignment
lidar_ids = {p.stem.split('_')[-1] for p in LIDAR_DIR.glob('lidar_patch_*.tif')}
s1_ids = {p.name.split('_')[-1] for p in S1_DIR.glob('s1_patch_*') if p.is_dir()}
paired_ids = sorted(lidar_ids & s1_ids)
assert paired_ids, 'No paired Sentinel-1/LiDAR patches found.'

def patch_centroid(patch_id):
    with rasterio.open(LIDAR_DIR / f'lidar_patch_{patch_id}.tif') as src:
        b = src.bounds
    return ((b.left + b.right) / 2.0, (b.bottom + b.top) / 2.0)

centroids = {pid: patch_centroid(pid) for pid in paired_ids}

def block_id_and_boundary_distance(cx, cy, block_size):
    bx, by = int(cx // block_size), int(cy // block_size)
    dx = min(cx - bx * block_size, (bx + 1) * block_size - cx)
    dy = min(cy - by * block_size, (by + 1) * block_size - cy)
    return (bx, by), min(dx, dy)

blocks = {}
dropped_buffer = []
for pid, (cx, cy) in centroids.items():
    bid, boundary_dist = block_id_and_boundary_distance(cx, cy, BLOCK_SIZE_M)
    if boundary_dist < BUFFER_M:
        dropped_buffer.append(pid)
        continue
    blocks.setdefault(bid, []).append(pid)

kept_total = sum(len(v) for v in blocks.values())
print(f'{len(dropped_buffer)} / {len(paired_ids)} patches dropped as boundary buffer')
print(f'{len(blocks)} spatial blocks remain, containing {kept_total} patches')

block_ids = list(blocks.keys())
random.Random(SEED).shuffle(block_ids)

target_val_patches = int(len(paired_ids) * VAL_FRACTION)
val_ids, train_ids = [], []
running_val_count = 0
for bid in block_ids:
    if running_val_count < target_val_patches:
        val_ids.extend(blocks[bid])
        running_val_count += len(blocks[bid])
    else:
        train_ids.extend(blocks[bid])

print(f'Spatial-block split: train={len(train_ids)}, val={len(val_ids)} '
      f'(target val={target_val_patches}, {len(dropped_buffer)} dropped as buffer)')

train_dataset = LidarEwDataset(S1_DIR, LIDAR_DIR, train_ids, CONTEXT_K, TARGET_HW, train=True)
val_dataset = LidarEwDataset(S1_DIR, LIDAR_DIR, val_ids, CONTEXT_K, TARGET_HW, train=False)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

### Verify the split is leakage-free

Same overlap-check as `08`/`09`/`10` -- must print `0 / N` before
training proceeds.

In [ ]:
# Overlap-gate check, identical to 08/09/10 -- hard-stops training if the split isn't actually leakage-free
from shapely.geometry import box
from shapely.strtree import STRtree

train_boxes = []
for pid in train_ids:
    with rasterio.open(LIDAR_DIR / f'lidar_patch_{pid}.tif') as src:
        train_boxes.append(box(*src.bounds))

val_boxes = []
for pid in val_ids:
    with rasterio.open(LIDAR_DIR / f'lidar_patch_{pid}.tif') as src:
        val_boxes.append((pid, box(*src.bounds)))

tree = STRtree(train_boxes)
overlap_count = 0
for pid, vbox in val_boxes:
    hits = tree.query(vbox)
    real_overlaps = [h for h in hits if train_boxes[h].intersects(vbox) and not train_boxes[h].touches(vbox)]
    if real_overlaps:
        overlap_count += 1
        print(f'Val patch {pid} still overlaps {len(real_overlaps)} training patch(es)')

print(f'\n{overlap_count} / {len(val_ids)} validation patches overlap a training patch (must be 0)')
assert overlap_count == 0, (
    'Spatial-block split still has leakage -- increase BLOCK_SIZE_M or BUFFER_M and re-run.'
)
print('Confirmed: spatial-block split is leakage-free. Safe to proceed to training.')

## Initialize the model and scheduler

In [ ]:
# Same architecture as 09/10 -- HH/HV repeats to 4 channels per view exactly like VV/VH did, so no shape changes needed
model = ConditionalUNet(in_channels=1, cond_channels=4 * CONTEXT_K, attr_dim=8 * CONTEXT_K, base_channels=128, embed_dim=256, unet_depth=4, attention_variant=ATTENTION_VARIANT, cond_k=CONTEXT_K).to(DEVICE)
scheduler = (LinearDiffusionScheduler(TIMESTEPS, device=DEVICE) if NOISE_SCHEDULE == 'linear' else CosineDiffusionScheduler(TIMESTEPS, device=DEVICE))
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
print('Trainable parameters:', sum(p.numel() for p in model.parameters() if p.requires_grad))

## Training loop

In [ ]:
# Training loop, identical to 09/10 -- masked MSE on the noised, demeaned LiDAR residual
from torch.cuda.amp import autocast, GradScaler

def masked_mse(prediction, target, mask):
    valid = mask.bool().unsqueeze(1)
    error = (prediction - target) ** 2
    return error[valid].mean()

scaler = GradScaler()
history = {'train_loss': [], 'val_loss': []}
best_val = float('inf')
for epoch in range(EPOCHS):
    model.train()
    train_total = 0.0
    for batch in train_loader:
        target = batch['lidar'].to(DEVICE, non_blocking=True)
        condition = batch['s1'].to(DEVICE, non_blocking=True)
        attrs = batch['attrs'].to(DEVICE, non_blocking=True)
        mask = batch['mask'].to(DEVICE, non_blocking=True)
        timestep = torch.randint(0, TIMESTEPS, (target.size(0),), device=DEVICE)
        optimizer.zero_grad(set_to_none=True)
        with autocast():
            noisy = scheduler.q_sample(target, timestep)
            prediction = model(noisy, condition, attrs, timestep)
            loss = masked_mse(prediction, target, mask)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        train_total += loss.item()
    model.eval()
    val_total = 0.0
    with torch.no_grad():
        for batch in val_loader:
            target = batch['lidar'].to(DEVICE, non_blocking=True)
            condition = batch['s1'].to(DEVICE, non_blocking=True)
            attrs = batch['attrs'].to(DEVICE, non_blocking=True)
            mask = batch['mask'].to(DEVICE, non_blocking=True)
            timestep = torch.randint(0, TIMESTEPS, (target.size(0),), device=DEVICE)
            with autocast():
                prediction = model(scheduler.q_sample(target, timestep), condition, attrs, timestep)
                val_total += masked_mse(prediction, target, mask).item()
    train_loss = train_total / max(1, len(train_loader))
    val_loss = val_total / max(1, len(val_loader))
    history['train_loss'].append(train_loss); history['val_loss'].append(val_loss)
    print(f'Epoch {epoch + 1:03d}/{EPOCHS}: train={train_loss:.6f} val={val_loss:.6f}')
    if val_loss < best_val:
        best_val = val_loss
        torch.save({'model_state_dict': model.state_dict(), 'config': {'context_k': CONTEXT_K, 'timesteps': TIMESTEPS, 'noise_schedule': NOISE_SCHEDULE, 'region': REGION}, 'epoch': epoch + 1, 'val_loss': val_loss}, CHECKPOINT_DIR / f's1_{REGION}_ew_realattrs_spatialsplit_unet_best.pth')

## Evaluate with Tessa's reconstruction metrics

In [ ]:
# Evaluation loop, identical metric suite to every other notebook in this project
best_path = CHECKPOINT_DIR / f's1_{REGION}_ew_realattrs_spatialsplit_unet_best.pth'
checkpoint = torch.load(best_path, map_location=DEVICE)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
sampler = p_sample_loop_ddim
metric_rows = []
example_patches = []
N_EXAMPLES = 6
with torch.no_grad():
    for batch in val_loader:
        target = batch['lidar'].to(DEVICE)
        condition = batch['s1'].to(DEVICE)
        attrs = batch['attrs'].to(DEVICE)
        mask = batch['mask'].to(DEVICE).bool()
        prediction = sampler(model, scheduler, target.shape, condition, attrs, DEVICE)
        means = batch['patch_mean'].to(DEVICE).view(-1, 1, 1, 1)
        gt_absolute = target + means
        pred_absolute = prediction + means
        for i, patch_id in enumerate(batch['patch_id']):
            gt_i, pred_i, mask_i = gt_absolute[i], pred_absolute[i], mask[i]
            gt_valid = gt_i.squeeze()[mask_i].cpu().numpy()
            pred_valid = pred_i.squeeze()[mask_i].cpu().numpy()
            row = {
                'patch_id': patch_id,
                'rmse_m': float(rmse(gt_i, pred_i, mask_i).item()),
                'bias_m': float(bias(gt_i, pred_i, mask_i).item()),
                'sigma_error_pct': float(sigma_error(gt_i, pred_i, mask_i).item()),
                'normal_angle_error_deg': float(normal_angle_error(gt_i, pred_i, mask_i, pixel_size=1.0, degrees=True).item()),
                'jsd': float(average_jsd_multiscale(gt_i, pred_i, pixel_size=1.0, mask=mask_i).item()),
                'psd_rmse': float(log_psd_rmse(gt_i, pred_i, pixel_size=1.0, mask=mask_i).item()),
                'zncc': float(zncc(gt_i, pred_i, mask_i).item()),
                'gt_std_val': float(gt_valid.std()) if gt_valid.size > 0 else float('nan'),
                'pred_std_val': float(pred_valid.std()) if pred_valid.size > 0 else float('nan'),
            }
            metric_rows.append(row)
            if len(example_patches) < N_EXAMPLES:
                example_patches.append({
                    'patch_id': patch_id,
                    'gt': gt_i.squeeze().cpu().numpy(),
                    'pred': pred_i.squeeze().cpu().numpy(),
                    'mask': mask_i.squeeze().cpu().numpy(),
                })
metrics_path = OUTPUT_DIR / 's1_ew_realattrs_spatialsplit_validation_metrics.json'
with metrics_path.open('w') as handle:
    json.dump(metric_rows, handle, indent=2)
print('Saved:', metrics_path)
print('Mean metrics:', {key: float(np.nanmean([row[key] for row in metric_rows])) for key in metric_rows[0] if key != 'patch_id'})

## Visual check: Sentinel-2 imagery alongside predictions

Michel's actual request, confirmed after he said he doesn't have
Tessa's checkpoint: not a model-vs-model comparison, just the raw
Sentinel-2 optical imagery displayed next to the SAR-based prediction,
so a human can visually judge whether the same surface features are
visible in both. No model of his is needed for this. Adapted from
`pcrtc/08`'s unseen-date test -- but unlike `08`, this fetches S2 near
the *actual* survey date (`2024-04-16`) rather than a shifted window,
since EW's own conditioning dates are already near-perfect matches to
the survey.

In [ ]:
# Fetch a low-cloud Sentinel-2 scene near the survey date, purely for visual comparison -- no model needed
import pystac_client
import planetary_computer
from dotenv import load_dotenv
from shapely.geometry import box as shapely_box, shape
from shapely.ops import unary_union
from rasterio.warp import transform_geom, transform_bounds
from rasterio.windows import from_bounds, transform as win_transform
from concurrent.futures import ThreadPoolExecutor, as_completed
import datetime as dt

load_dotenv()
if os.environ.get('PC_SDK_SUBSCRIPTION_KEY'):
    planetary_computer.settings.set_subscription_key(os.environ['PC_SDK_SUBSCRIPTION_KEY'])

def aoi_from_lidar_patches(patches_dir, max_files=300, workers=8):
    paths = sorted(patches_dir.glob('lidar_patch_*.tif'))
    if len(paths) > max_files:
        stride = len(paths) / max_files
        paths = [paths[int(i * stride)] for i in range(max_files)]
    def read_bounds(path):
        with rasterio.open(path) as src:
            return src.crs, src.bounds
    results = []
    with ThreadPoolExecutor(max_workers=workers) as pool:
        futures = [pool.submit(read_bounds, p) for p in paths]
        for future in as_completed(futures):
            results.append(future.result())
    crs = results[0][0]
    native = unary_union([shapely_box(*bounds) for _, bounds in results])
    geojson = transform_geom(crs, 'EPSG:4326', native.__geo_interface__)
    return shape(geojson).buffer(0)

aoi = aoi_from_lidar_patches(LIDAR_DIR)
aoi_ll = aoi.convex_hull

SEARCH_DAYS_S2 = 30
catalog = pystac_client.Client.open(
    'https://planetarycomputer.microsoft.com/api/stac/v1',
    modifier=planetary_computer.sign_inplace,
)
start = LIDAR_SURVEY_DATE - dt.timedelta(days=SEARCH_DAYS_S2)
end = LIDAR_SURVEY_DATE + dt.timedelta(days=SEARCH_DAYS_S2)

s2_items = sorted(catalog.search(
    collections=['sentinel-2-l2a'], intersects=aoi_ll.__geo_interface__,
    datetime=f'{start.isoformat()}/{end.isoformat()}',
    query={'eo:cloud_cover': {'lt': 30}},
).items(), key=lambda it: it.properties.get('eo:cloud_cover', 100))
print(f'S2 scenes found near survey date (cloud<30%): {len(s2_items)}')

s2_item = s2_items[0] if s2_items else None
if s2_item:
    print('Using S2 scene:', s2_item.datetime.date(), '| cloud cover:', s2_item.properties.get('eo:cloud_cover'),
          '| days from survey:', (s2_item.datetime.date() - LIDAR_SURVEY_DATE).days)
else:
    print('No usable S2 scene found in this window -- widen SEARCH_DAYS_S2 or relax the cloud filter.')

In [ ]:
# Save an RGB visual mosaic of the AOI, then crop it to each example patch's exact footprint
EW_QC_DIR = WORKING_REPO / 'raw_data' / 'tuk_ew_s2_visual_check'
EW_QC_DIR.mkdir(parents=True, exist_ok=True)
S2_MOSAIC_PATH = EW_QC_DIR / 's2_visual_mosaic.tif'

if s2_item is not None:
    with rasterio.open(s2_item.assets['visual'].href) as src:
        aoi_bounds = transform_bounds('EPSG:4326', src.crs, *aoi_ll.bounds)
        window = from_bounds(*aoi_bounds, transform=src.transform)
        rgb = src.read([1, 2, 3], window=window)
        out_transform = win_transform(window, src.transform)
        meta = {'driver': 'GTiff', 'count': 3, 'height': rgb.shape[1], 'width': rgb.shape[2],
                'dtype': rgb.dtype, 'crs': src.crs, 'transform': out_transform}
        with rasterio.open(S2_MOSAIC_PATH, 'w', **meta) as dst:
            dst.write(rgb)
    print('Wrote S2 visual mosaic:', S2_MOSAIC_PATH)


def extract_s2_patch(patch_id, s2_mosaic_path, lidar_dir):
    lidar_path = lidar_dir / f'lidar_patch_{patch_id}.tif'
    with rasterio.open(lidar_path) as lsrc:
        lidar_bounds, lidar_crs = lsrc.bounds, lsrc.crs
        out_h, out_w = lsrc.height, lsrc.width
    with rasterio.open(s2_mosaic_path) as src:
        bounds = transform_bounds(lidar_crs, src.crs, *lidar_bounds, densify_pts=21)
        window = from_bounds(*bounds, transform=src.transform)
        rgb = src.read([1, 2, 3], window=window, out_shape=(3, out_h, out_w))
    return np.moveaxis(rgb, 0, -1)  # [H, W, 3] for imshow


ew_s2_crops = {}
if S2_MOSAIC_PATH.exists():
    for ex in example_patches:
        pid = ex['patch_id']
        try:
            ew_s2_crops[pid] = extract_s2_patch(pid, S2_MOSAIC_PATH, LIDAR_DIR)
        except Exception as exc:
            print(f'Could not extract S2 crop for patch {pid}: {exc}')
    print(f'Extracted S2 crops for {len(ew_s2_crops)} / {len(example_patches)} example patches')
else:
    print('No S2 mosaic available -- skipping visual crops (S2 row will be blank in the figure).')

In [ ]:
import matplotlib.pyplot as plt

# S2 imagery / GT / Pred / Error grid for the same example patches evaluated above
n_show = min(6, len(example_patches))
fig, axes = plt.subplots(4, n_show, figsize=(4 * n_show, 16), squeeze=False)
for col, ex in enumerate(example_patches[:n_show]):
    gt_c = ex['gt'] - ex['gt'][ex['mask']].mean()
    pred_c = ex['pred'] - ex['pred'][ex['mask']].mean()
    pid = ex['patch_id']
    axes[0, col].set_title(f'Patch {pid}', fontweight='bold')
    if pid in ew_s2_crops:
        axes[0, col].imshow(ew_s2_crops[pid])
    axes[0, col].axis('off')
    axes[1, col].imshow(gt_c, cmap='RdBu_r'); axes[1, col].axis('off')
    axes[2, col].imshow(pred_c, cmap='RdBu_r'); axes[2, col].axis('off')
    axes[3, col].imshow(pred_c - gt_c, cmap='seismic'); axes[3, col].axis('off')
for row, label in enumerate(['S2 imagery (visual check)', 'GT LiDAR (centered)', 'Pred, EW-mode input (centered)', 'Error']):
    axes[row, 0].text(-0.25, 0.5, label, ha='right', va='center', transform=axes[row, 0].transAxes, fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 's1_ew_realattrs_s2_visual_check.png', dpi=150, bbox_inches='tight')
print('Saved:', OUTPUT_DIR / 's1_ew_realattrs_s2_visual_check.png')
plt.show()

## Comparison protocol

Compare directly against `09`'s in-region IW/PC-RTC numbers
(`s1_pcrtc_realattrs_spatialsplit_validation_metrics.json`) -- same
validation patches (identical spatial-block split), so any difference
is attributable to the conditioning data source (EW/HH+HV/40m vs
IW/VV+VH/10m), not a different experimental setup. Given the resolution
confound noted at the top of this notebook, a clean win either way
still can't fully separate "polarization informativeness" from
"resolution" as the cause -- worth stating explicitly rather than
overclaiming either interpretation.

In [ ]:
# Compare against 09's in-region IW/PC-RTC numbers on the identical validation split
iw_metrics_path = OUTPUT_DIR / 's1_pcrtc_realattrs_spatialsplit_validation_metrics.json'
new_mean = {key: float(np.nanmean([row[key] for row in metric_rows])) for key in metric_rows[0] if key != 'patch_id'}

if iw_metrics_path.exists():
    iw_rows = json.load(open(iw_metrics_path))
    iw_mean = {k: float(np.nanmean([r[k] for r in iw_rows])) for k in iw_rows[0] if k != 'patch_id'}
    print(f'{"metric":<20}{"IW/PC-RTC (09)":>20}{"EW (this notebook)":>22}')
    for k in new_mean:
        if k in iw_mean:
            print(f'{k:<20}{iw_mean[k]:>20.4f}{new_mean[k]:>22.4f}')
else:
    print('09 metrics file not found -- compare manually against the documented 09 numbers.')